<a href="https://colab.research.google.com/github/shreyanshsamaje7-prog/AI_LAB/blob/main/tiles.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import heapq

def misplaced_tiles_heuristic(state, goal_state):
    """Counts how many tiles are not in their goal positions (excluding the blank tile 0)."""
    return sum(1 for i in range(9) if state[i] != 0 and state[i] != goal_state[i])

def get_neighbors(state):
    """Generates valid successor states by moving the blank tile (0)."""
    neighbors = []
    zero_idx = state.index(0)
    row, col = divmod(zero_idx, 3)

    # Possible moves: Up, Down, Left, Right
    moves = [(-1, 0), (1, 0), (0, -1), (0, 1)]

    for dr, dc in moves:
        r, c = row + dr, col + dc
        if 0 <= r < 3 and 0 <= c < 3:
            new_zero_idx = r * 3 + c
            state_list = list(state)
            state_list[zero_idx], state_list[new_zero_idx] = state_list[new_zero_idx], state_list[zero_idx]
            neighbors.append(tuple(state_list))

    return neighbors

def is_solvable(start_state, goal_state):
    """
    Checks if start state can reach goal state by comparing inversion counts.
    An 8-puzzle is solvable if and only if both states have the same inversion parity.
    """
    def count_inversions(state):
        arr = [x for x in state if x != 0]
        inversions = 0
        for i in range(len(arr)):
            for j in range(i + 1, len(arr)):
                if arr[i] > arr[j]:
                    inversions += 1
        return inversions

    return (count_inversions(start_state) % 2) == (count_inversions(goal_state) % 2)

def solve_a_star(start_state, goal_state):
    """Solves the 8-puzzle using A* search algorithm."""
    if not is_solvable(start_state, goal_state):
        return None, 0

    h_start = misplaced_tiles_heuristic(start_state, goal_state)
    pq = [(h_start, 0, start_state, [start_state])]
    visited = {start_state: 0}

    while pq:
        f, g, current, path = heapq.heappop(pq)

        if current == goal_state:
            return path, len(visited)

        if g > visited[current]:
            continue

        for neighbor in get_neighbors(current):
            tentative_g = g + 1
            if neighbor not in visited or tentative_g < visited[neighbor]:
                visited[neighbor] = tentative_g
                h = misplaced_tiles_heuristic(neighbor, goal_state)
                f_score = tentative_g + h
                heapq.heappush(pq, (f_score, tentative_g, neighbor, path + [neighbor]))

    return None, len(visited)

def print_board(state):
    """Prints a 1D tuple as a formatted 3x3 grid."""
    for i in range(0, 9, 3):
        row = [str(x) if x != 0 else " " for x in state[i:i+3]]
        print("[" + " ".join(row) + "]")
    print()

def get_user_input(prompt_label):
    """Parses and validates 9 space-separated integers from the user."""
    print(f"\nEnter the {prompt_label} state (9 numbers separated by spaces, 0 for blank):")
    print("Example: 1 2 3 4 0 6 7 5 8")
    while True:
        try:
            user_input = input(">> ").strip().split()
            numbers = tuple(map(int, user_input))

            if len(numbers) != 9:
                print("Error: You must enter exactly 9 numbers.")
                continue
            if set(numbers) != set(range(9)):
                print("Error: Input must contain all digits from 0 to 8 without duplicates.")
                continue

            return numbers
        except ValueError:
            print("Error: Invalid input. Please enter integers only.")

# Interactive Execution
if __name__ == "__main__":
    start_state = get_user_input("INITIAL")
    goal_state = get_user_input("GOAL")

    print("\nInitial Board:")
    print_board(start_state)
    print("Goal Board:")
    print_board(goal_state)

    path, nodes_visited = solve_a_star(start_state, goal_state)

    if path:
        print(f"Goal reached in {len(path) - 1} moves!")
        print(f"Total states explored: {nodes_visited}\n")
        print("Solution Steps:")
        for step, state in enumerate(path):
            print(f"Step {step}:")
            print_board(state)
    else:
        print("No solution exists for this pair of initial and goal states.")


Enter the INITIAL state (9 numbers separated by spaces, 0 for blank):
Example: 1 2 3 4 0 6 7 5 8
>> 2 8 3 1 6 4 0 7 5

Enter the GOAL state (9 numbers separated by spaces, 0 for blank):
Example: 1 2 3 4 0 6 7 5 8
>> 1 2 3 8 0 4 7 6 5

Initial Board:
[2 8 3]
[1 6 4]
[  7 5]

Goal Board:
[1 2 3]
[8   4]
[7 6 5]

Goal reached in 6 moves!
Total states explored: 17

Solution Steps:
Step 0:
[2 8 3]
[1 6 4]
[  7 5]

Step 1:
[2 8 3]
[1 6 4]
[7   5]

Step 2:
[2 8 3]
[1   4]
[7 6 5]

Step 3:
[2   3]
[1 8 4]
[7 6 5]

Step 4:
[  2 3]
[1 8 4]
[7 6 5]

Step 5:
[1 2 3]
[  8 4]
[7 6 5]

Step 6:
[1 2 3]
[8   4]
[7 6 5]

